# Phishing URL Detector Prototype

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import re
import pandas as pd
from urllib.parse import urlparse

# --- Step 1: Feature Extraction ---
def extract_features(url):
    parsed = urlparse(url)
    return {
        'url': url,
        'url_length': len(url),
        'has_ip': bool(re.match(r'http[s]?://(\d{1,3}\.){3}\d{1,3}', url)),
        'has_at_symbol': '@' in url,
        'has_dash': '-' in parsed.netloc,
        'subdomain_count': len(parsed.netloc.split('.')) - 2,
        'is_https': parsed.scheme == 'https',
        'has_suspicious_words': any(word in url.lower() for word in ['login', 'verify', 'secure', 'update', 'bank']),
        'uses_punycode': 'xn--' in parsed.netloc
    }

# --- Step 2: Define Risk Scoring ---
def risk_score(features):
    score = 0
    score += features['has_ip'] * 2
    score += features['has_at_symbol'] * 2
    score += features['has_dash']
    score += features['subdomain_count'] > 2
    score += not features['is_https']
    score += features['has_suspicious_words'] * 2
    score += features['uses_punycode'] * 2
    return score

# --- Step 3: Sample URLs ---
test_urls = [
    'http://192.168.1.1/login',
    'https://www.paypal.com',
    'http://paypal-login.com',
    'https://xn--pple-43d.com',
    'https://secure-update-account.com',
    'https://www.google.com',
    'http://example.com@evil.com',
    'https://login-bank-verification.com',
    'http://normal-website.org',
    'https://very.safe.site.com'
]

# --- Step 4: Run Detector ---
df = pd.DataFrame([extract_features(url) for url in test_urls])
df['risk_score'] = df.apply(risk_score, axis=1)
df = df.sort_values('risk_score', ascending=False)
df.reset_index(drop=True, inplace=True)

# --- Step 5: Display Results ---
from IPython.display import display
display(df)